### DATA ANALYSIS

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Dataset location
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "segments.csv"
)

# Check file exists
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

# Load original telemetry
df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Successfully loaded OPSSAT-AD")

print("Dataset shape:", df.shape)

display(df.head(10))

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(
    df.dtypes.to_frame("dtype")
)

print("\nMissing values:")
display(
    df.isna().sum().to_frame("missing_values")
)

print("\nUnique channels:")
print(df["channel"].nunique())

print(df["channel"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))

print("\nOfficial training/test flag:")
print(df["train"].value_counts(dropna=False))

print("Textual label distribution:")
display(df["label"].value_counts(dropna=False))

print("\nNumerical anomaly distribution:")
display(df["anomaly"].value_counts(dropna=False))

print("\nRelationship between the two:")
display(
    pd.crosstab(
        df["label"],
        df["anomaly"]
    )
)

In [ ]:
segment_channel_counts = (
    df.groupby("segment")["channel"].nunique()
)

print(
    "Unique segment IDs:",
    df["segment"].nunique()
)

print(
    "Segment IDs associated with multiple channels:",
    (segment_channel_counts > 1).sum()
)

assert segment_channel_counts.max() == 1

#### Reconstruct the individual telemetry segments

In [ ]:
# Convert numerical values
df["value_numeric"] = pd.to_numeric(
    df["value"],
    errors="coerce"
)

# Parse timestamps
df["timestamp_parsed"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

# Convert sampling information where possible
df["sampling_numeric"] = pd.to_numeric(
    df["sampling"],
    errors="coerce"
)

segment_summary = (

    df.groupby("segment")

    .agg(

        channel=("channel", "first"),

        n_samples=("value", "size"),

        n_missing_values=(
            "value",
            lambda x: x.isna().sum()
        ),

        # Textual label: retained for auditing only
        text_label=("label", "first"),

        # Actual numerical anomaly target
        anomaly=("anomaly", "first"),

        n_unique_anomaly=("anomaly", "nunique"),

        train=("train", "first"),

        n_unique_train=("train", "nunique"),

        sampling=("sampling", "first"),

        n_unique_sampling=("sampling", "nunique"),

        timestamp_start=("timestamp_parsed", "min"),

        timestamp_end=("timestamp_parsed", "max"),

        sampling_min=("sampling_numeric", "min"),

        sampling_max=("sampling_numeric", "max")

    )

    .reset_index()

)

print("Total reconstructed segments:", len(segment_summary))

display(segment_summary.head(10))

In [ ]:
inconsistent_segments = segment_summary[
    (segment_summary["n_unique_anomaly"] > 1)
    |
    (segment_summary["n_unique_train"] > 1)
]

print(
    "Segments with inconsistent labels or split assignments:",
    len(inconsistent_segments)
)

display(inconsistent_segments.head(10))

print(
    "\nMissing telemetry values:",
    df["value_numeric"].isna().sum()
)

print(
    "Invalid timestamps:",
    df["timestamp_parsed"].isna().sum()
)

print(
    "Segments with inconsistent anomaly labels:",
    (segment_summary["n_unique_anomaly"] > 1).sum()
)

print(
    "Segments with inconsistent train assignments:",
    (segment_summary["n_unique_train"] > 1).sum()
)

print(
    "Segments with inconsistent sampling values:",
    (segment_summary["n_unique_sampling"] > 1).sum()
)

print(
    "Unique anomaly labels:",
    sorted(segment_summary["anomaly"].unique())
)

assert segment_summary["n_unique_anomaly"].eq(1).all()

assert segment_summary["n_unique_train"].eq(1).all()

assert set(segment_summary["anomaly"].unique()) == {0, 1}

assert set(segment_summary["train"].unique()) == {0, 1}

print("\nSegment validation completed.")

In [ ]:
global_distribution = pd.crosstab(
    segment_summary["train"],
    segment_summary["anomaly"]
)

global_distribution = global_distribution.reindex(
    index=[1, 0],
    columns=[0, 1],
    fill_value=0
)

global_distribution.index = [
    "Training",
    "Test"
]

global_distribution.columns = [
    "Nominal",
    "Anomalous"
]

display(global_distribution)

print("\nTotal segments:", len(segment_summary))

In [ ]:
assert global_distribution.loc["Training", "Nominal"] == 1273

assert global_distribution.loc["Training", "Anomalous"] == 321

assert global_distribution.loc["Test", "Nominal"] == 416

assert global_distribution.loc["Test", "Anomalous"] == 113

print("Official benchmark split reproduced successfully.")

In [ ]:
channel_distribution = pd.crosstab(
    segment_summary["channel"],
    segment_summary["anomaly"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

channel_distribution.columns = [
    "n_nominal",
    "n_anomalous"
]

channel_distribution["n_total"] = (
    channel_distribution["n_nominal"]
    + channel_distribution["n_anomalous"]
)

channel_distribution["anomaly_rate"] = (
    channel_distribution["n_anomalous"]
    / channel_distribution["n_total"]
)

display(
    channel_distribution.sort_values(
        "n_total",
        ascending=False
    ).round(3)
)

In [ ]:
channel_split_distribution = pd.crosstab(
    [
        segment_summary["channel"],
        segment_summary["train"]
    ],
    segment_summary["anomaly"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

channel_split_distribution.columns = [
    "nominal",
    "anomalous"
]

display(channel_split_distribution)

In [ ]:
sampling_summary = (
    segment_summary
    .groupby("channel")
    .agg(
        n_segments=("segment", "count"),
        min_sampling=("sampling_min", "min"),
        max_sampling=("sampling_max", "max"),
        median_sampling=("sampling", "median"),
        inconsistent_sampling_segments=(
            "n_unique_sampling",
            lambda x: (x > 1).sum()
        )
    )
)

display(sampling_summary)

This means that sampling is constant within each segment, but differs between segments.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for channel in ["CADC0872", "CADC0873", "CADC0874"]:

    values = segment_summary.loc[
        segment_summary["channel"] == channel,
        "n_samples"
    ]

    ax.hist(
        values,
        bins=25,
        alpha=0.4,
        label=channel
    )

ax.set_title("Telemetry segment lengths")

ax.set_xlabel("Number of measurements per segment")

ax.set_ylabel("Number of segments")

ax.legend()

ax.grid(alpha=0.2)

plt.tight_layout()

figures_dir = PROJECT_ROOT / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    figures_dir / "segment_length_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

#### Sampling-rate investigation
Changes in sampling frequency

In [ ]:
selected_channel = "CADC0872"

channel_data = segment_summary[
    segment_summary["channel"] == selected_channel
].copy()

sampling_distribution = pd.crosstab(
    [
        channel_data["sampling"],
        channel_data["train"]
    ],
    channel_data["anomaly"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

sampling_distribution.columns = [
    "nominal",
    "anomalous"
]

display(sampling_distribution)

In [ ]:
def inspect_segment_timing(segment_id):

    segment_df = (
        df[df["segment"] == segment_id]
        .sort_values("timestamp_parsed")
    )

    time_differences = (
        segment_df["timestamp_parsed"]
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    expected_sampling = segment_df["sampling"].iloc[0]

    return {
        "segment": segment_id,

        "sampling_seconds": expected_sampling,

        "median_actual_interval":
            time_differences.median(),

        "minimum_actual_interval":
            time_differences.min(),

        "maximum_actual_interval":
            time_differences.max(),

        "proportion_matching_sampling":
            np.isclose(
                time_differences,
                expected_sampling
            ).mean()
    }

In [ ]:
example_segments = (
    channel_data
    .groupby("sampling")["segment"]
    .first()
    .tolist()
)

timing_results = pd.DataFrame([
    inspect_segment_timing(segment_id)
    for segment_id in example_segments
])

display(timing_results.round(3))

#### Visualise actual telemetry

In [ ]:
selected_sampling = int(
    channel_data["sampling"].mode().iloc[0]
)

print("Selected sampling interval:", selected_sampling)

training_segments = channel_data[
    (channel_data["train"] == 1) &
    (channel_data["sampling"] == selected_sampling)
].copy()

print(
    training_segments["anomaly"]
    .value_counts()
)

In [ ]:
representative_segments = []

for anomaly_label in [0, 1]:

    candidates = (
        training_segments[
            training_segments["anomaly"] == anomaly_label
        ]
        .sort_values("n_samples")
        .reset_index(drop=True)
    )

    if candidates.empty:
        raise ValueError(
            f"No examples for anomaly={anomaly_label}"
        )

    selected = candidates.iloc[
        len(candidates) // 2
    ]

    representative_segments.append(selected)

display(
    pd.DataFrame(representative_segments)[
        [
            "segment",
            "anomaly",
            "sampling",
            "n_samples",
            "timestamp_start",
            "timestamp_end"
        ]
    ]
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(13, 7)
)

titles = {
    0: "Nominal telemetry",
    1: "Anomalous telemetry"
}

for ax, selected in zip(
    axes,
    representative_segments
):

    segment_id = selected["segment"]

    segment_df = (
        df[df["segment"] == segment_id]
        .sort_values("timestamp_parsed")
        .copy()
    )

    elapsed_seconds = (
        segment_df["timestamp_parsed"]
        - segment_df["timestamp_parsed"].iloc[0]
    ).dt.total_seconds()

    ax.plot(
        elapsed_seconds,
        segment_df["value_numeric"],
        linewidth=0.9
    )

    ax.set_title(
        f"{titles[selected['anomaly']]} "
        f"(Segment {segment_id})"
    )

    ax.set_xlabel("Elapsed time (seconds)")

    ax.set_ylabel("Magnetometer reading")

    ax.grid(alpha=0.2)

fig.suptitle(
    f"OPS-SAT telemetry — {selected_channel}",
    fontsize=14
)

plt.tight_layout()

figures_dir = PROJECT_ROOT / "reports" / "figures"

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)

fig.savefig(
    figures_dir / "opssat_representative_telemetry.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()